# Practice 07 — MLOps: Monitoring & Drift

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/theDocWho/lbv-notebooks/blob/main/ai-ml/07-mlops-drift.ipynb) [![Open in Kaggle](https://img.shields.io/badge/Kaggle-open-20BEFF?logo=kaggle&logoColor=white)](https://kaggle.com/kernels/welcome?src=https://raw.githubusercontent.com/theDocWho/lbv-notebooks/main/ai-ml/07-mlops-drift.ipynb)

Companion drills for **Phases 7–8** of [Learn by Visualization](https://learn-by-visuallization.org/illustrated/index.html) —
pairs with the [MLOps lifecycle & drift](https://learn-by-visuallization.org/illustrated/7-mlops/mlops-lifecycle.html) and
[agent patterns](https://learn-by-visuallization.org/illustrated/8-agentic/agent-patterns.html) explainers.

**How this works:** each exercise is a `# TODO` scaffold followed by a self-check cell. Fill in the TODO, re-run both cells, and turn the ❌ into a ✅. No answers are hidden in the notebook — the checks themselves tell you what "correct" means. Runs on local Jupyter, Google Colab, or Kaggle (free, no setup, no GPU needed).

In [ ]:
NEEDED = [("numpy", "numpy"), ("sklearn", "scikit-learn")]

# ---- setup: run me first ----
import importlib.util, subprocess, sys
for pkg, pipname in NEEDED:
    if importlib.util.find_spec(pkg) is None:
        print(f"installing {pipname} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pipname])

RESULTS = {}
def check(name, fn, hint=""):
    """Self-check: PASS/FAIL with a hint. Never raises -- rerun the cell after editing your answer."""
    try:
        ok = bool(fn())
    except Exception as e:
        ok, hint = False, f"{hint} (your code raised {type(e).__name__}: {e})"
    RESULTS[name] = ok
    print(("\u2705 PASS" if ok else "\u274c FAIL") + f" \u2014 {name}" + ("" if ok else f"\n   hint: {hint}"))

import numpy as np
rng = np.random.default_rng(0)


### Exercise 1 — a baseline you can monitor
Train a `LogisticRegression` on `Xtr, ytr` (given) and record `base_acc` on the clean test set.
Every monitoring story starts with "what was normal?".

*Hint: fit, then `.score(Xte, yte)`.*

In [ ]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=1000, n_features=6, n_informative=4, class_sep=2.0, random_state=0)
Xtr, Xte, ytr, yte = train_test_split(X, y, random_state=0)

model    = None  # TODO: fitted LogisticRegression
base_acc = None  # TODO


In [ ]:
check("baseline accuracy recorded (> 0.75)",
      lambda: base_acc is not None and float(base_acc) > 0.75, "fit on train, score on test")

### Exercise 2 — simulate covariate drift, watch accuracy fall
`X_drift` shifts two features by +2σ (the world changed; labels didn't). Compute `drift_acc` with
the SAME model — no retraining. The check asserts performance drops: this silent decay is why
deployed models need monitors.

*Hint: `model.score(X_drift, yte)`.*

In [ ]:
X_drift = Xte.copy()
X_drift[:, 0] += 2 * Xte[:, 0].std()
X_drift[:, 2] += 2 * Xte[:, 2].std()

drift_acc = None  # TODO


In [ ]:
check("accuracy degrades under drift",
      lambda: drift_acc is not None and drift_acc < base_acc - 0.03,
      "same model, shifted inputs — the decision boundary is now in the wrong place")

### Exercise 3 — detect it WITHOUT labels: PSI
In production you rarely have fresh labels — you detect drift from the **inputs**. Implement the
Population Stability Index for one feature: bin the reference values into 10 quantile bins, get each
bin's share in reference vs current, and sum `(cur − ref) · ln(cur/ref)`.
Rule of thumb: PSI < 0.1 stable · 0.1–0.25 shifting · > 0.25 act.

*Hint: bins from `np.quantile(ref, np.linspace(0, 1, 11))`; shares via `np.histogram(x, bins)[0] / len(x)`;
add 1e-6 to both shares before the log.*

In [ ]:
def psi(ref, cur, n_bins=10):
    # TODO
    ...


In [ ]:
_f0_ref, _f0_cur = Xte[:, 0], X_drift[:, 0]
check("identical distributions -> PSI ~ 0",
      lambda: psi(_f0_ref, _f0_ref) < 1e-6, "no movement between bins")
check("the drifted feature screams (PSI > 0.25)",
      lambda: psi(_f0_ref, _f0_cur) > 0.25, "a 2σ shift empties the low bins into the high ones")
check("an undrifted feature stays quiet (PSI < 0.1)",
      lambda: psi(Xte[:, 1], X_drift[:, 1]) < 0.1, "feature 1 was never shifted")

### Exercise 4 — the monitor
Write `monitor(ref_X, cur_X)`: return the sorted list of feature indices whose PSI > 0.25 — the
page an on-call engineer actually wants.

*Hint: loop features, reuse psi().*

In [ ]:
def monitor(ref_X, cur_X, threshold=0.25):
    # TODO: indices of drifted features, sorted
    ...


In [ ]:
check("exactly the two shifted features are flagged",
      lambda: monitor(Xte, X_drift) == [0, 2],
      "features 0 and 2 got the +2σ treatment; everything else should pass")

### Stretch goal — close the loop
Retrain the model on a mix of old + drifted data and confirm accuracy recovers. Then sketch (in a
markdown cell) the full loop: monitor → alert → collect labels → retrain → redeploy → monitor. That
loop IS MLOps — compare with the
[lifecycle explainer](https://learn-by-visuallization.org/illustrated/7-mlops/mlops-lifecycle.html).

In [ ]:
# stretch — your playground


In [ ]:
# ---- self-score ----
passed, total = sum(RESULTS.values()), len(RESULTS)
print(f"Self-score: {passed}/{total} checks passing")
print("\U0001f389 All green \u2014 move on to the next explainer!" if total and passed == total
      else "Rerun each check cell as you fill in the TODOs above.")
